In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

from riskcal.analysis import get_beta_from_adp, get_beta_from_zcdp, get_beta_from_gdp, get_advantage_from_gdp
from dpmm.models.base.mechanisms import cdp_rho
from mst import mu_from_eps_delta
from audit_utils import run_audit, mu_lower_from_two_groups

In [ ]:
EPSILON = 1.0
DELTA = 1e-2

In [ ]:
N_TRAIN = 2000
N_VALID = 1000
N_TEST = 2000

In [ ]:
THEORY_RHO= cdp_rho(EPSILON, DELTA)
IMPLIED_MU = np.sqrt(2*THEORY_RHO)
print(f"Implied mu: {IMPLIED_MU} <---")


THEORY_MU = mu_from_eps_delta(EPSILON, DELTA)
print(f"Theory mu: {THEORY_MU}")


In [ ]:
with open('../data/features.pkl', 'rb') as handle:
    features = pickle.load(handle)

In [ ]:
default_results = run_audit(features["out"],
                            features["in"],
                            n_train=N_TRAIN,
                            n_valid=N_VALID,
                            n_test=N_TEST,
                            random_state=13)

In [ ]:
print(f"Empirical mu: {default_results['test']['point']['mu_lower']} <--")


In [ ]:
def adp_frontier_from_eps_delta(epsilon, delta, n_points=500):
    """
    Returns theoretical ADP frontier curve
    """
    clip_eps = 1e-6
    alpha = np.linspace(clip_eps, 1 - clip_eps, n_points)
    beta = get_beta_from_adp(epsilon, delta, alpha)
    return alpha, beta


def zcdp_frontier_from_rho(rho, n_points=500):
    """
    Returns theoretical zCDP frontier curve
    """
    clip_eps = 1e-6
    alpha = np.linspace(clip_eps, 1 - clip_eps, n_points)
    beta = get_beta_from_zcdp(rho, alpha)
    return alpha, beta


def gdp_frontier_from_mu(mu, n_points=500):
    """
    Returns theoretical GDP frontier curve
    """
    clip_eps = 1e-6
    alpha = np.linspace(clip_eps, 1 - clip_eps, n_points)
    beta = get_beta_from_gdp(mu, alpha)
    return alpha, beta


In [ ]:
val_curve = default_results["valid"]["curve"]
fpr = val_curve["FPR"]
fnr = val_curve["FNR"]

In [ ]:
alpha_th_eps, beta_th_eps = adp_frontier_from_eps_delta(EPSILON, DELTA)
alpha_th_rho, beta_th_rho = zcdp_frontier_from_rho(THEORY_RHO)
alpha_th_mu_imp, beta_th_mu_imp = gdp_frontier_from_mu(IMPLIED_MU)
alpha_th_mu, beta_th_mu = gdp_frontier_from_mu(THEORY_MU)

In [ ]:
fig = plt.figure(figsize=(6,6))
plt.gca().set_aspect('equal', adjustable='box')

# Empirical audit (primary)
plt.plot(
    fpr, fnr,
    color="black",
    linewidth=3.0,
    alpha=0.95,
    label="Empirical audit"
)

# μ-GDP via zCDP (primary theory)
plt.plot(
    alpha_th_mu_imp, beta_th_mu_imp,
    color="red",
    linewidth=3.0,
    alpha=0.95,
    label=r"$\mu$-GDP (via $\rho$-zCDP)"
)

# μ-GDP direct (secondary theory)
plt.plot(
    alpha_th_mu, beta_th_mu,
    color="red",
    linestyle="--",
    linewidth=2.0,
    alpha=0.65,
    label=r"$\mu$-GDP (via $(\epsilon,\delta)$-DP)"
)

# zCDP frontier (context)
plt.plot(
    alpha_th_rho, beta_th_rho,
    color="royalblue",
    linestyle="-.",
    linewidth=2.0,
    alpha=0.65,
    label=r"$\rho$-zCDP (context)"
)

# (ε,δ)-DP frontier (context)
plt.plot(
    alpha_th_eps, beta_th_eps,
    color="gray",
    linestyle=":",
    linewidth=2.0,
    alpha=0.5,
    label=r"$(\epsilon,\delta)$-DP (context)"
)

# 45-degree random-guess baseline: β = 1 − α
alpha_diag = np.linspace(0, 1, 200)
plt.plot(
    alpha_diag, 1 - alpha_diag,
    color="gray",
    linestyle="--",
    linewidth=2.0,
    alpha=0.5,
    label="Random guess"
)

plt.xlabel("FPR (α)", fontsize=12)
plt.ylabel("FNR (β)", fontsize=12)
plt.xlim(0, 1)
plt.ylim(0, 1)

plt.legend(loc="upper right", fontsize=12)

plt.grid(alpha=0.12)
plt.tight_layout()
plt.show()
# fig.savefig("../data/tradeoff.pdf")

In [ ]:
t = val_curve["thresholds"]
fpr = val_curve["FPR"]
fnr = val_curve["FNR"]
tpr = 1 - fnr
adv = val_curve["advantage"]
mu_hat = val_curve["mu_hat"]

# selected threshold index
t_sel = val_curve["opt_t"]
idx_sel = int(np.argmin(np.abs(t - t_sel)))

In [ ]:
# --- Figure & axes ---
fig, ax = plt.subplots()

# FORCE square plotting box
# ax.set_box_aspect(1)

# --- Baseline ---
# ax.axhline(y=0.0, color="lightgray", linestyle=":", linewidth=1.5, label="Random guess (adv.=0)")

# --- FPR / FNR ---
ax.plot(t, fpr, color="tab:blue", linewidth=2, alpha=0.65, label="FPR (α)")

ax.plot(t, fnr, color="tab:orange", linewidth=2, alpha=0.65, label="FNR (β)")

# --- Advantage (primary signal) ---
ax.plot(t, adv, color="darkgreen", linewidth=3, alpha=0.95, label="Empirical advantage")

# --- Theory reference ---
adv_theory = get_advantage_from_gdp(IMPLIED_MU)

ax.axhline(y=adv_theory, color="gray", linestyle="-.", linewidth=2, alpha=0.5, label=r"Theory advantage ($\mu$-GDP)")

# --- Selected threshold ---
ax.axvline(t_sel, color="black", linestyle="--", linewidth=2, alpha=0.95, label=r"Selected $\tau^*$")

# Star marker
# ax.scatter([t_sel], [adv[idx_sel]], s=160, marker="*", color="black", zorder=5)

# Annotation
# ax.text(t_sel + 0.015, adv[idx_sel], rf"$\tau^\star={t_sel:.2f}$", fontsize=11, va="center")

# --- Axes styling ---
ax.set_xlabel(r"Threshold $\tau$", fontsize=12)
ax.set_ylabel("Rate", fontsize=12)

ax.set_xlim(0,1)
ax.set_ylim(0,1)

# --- Legend (inside, clean) ---
handles, labels = ax.get_legend_handles_labels()
order = [
    labels.index(r"Selected $\tau^*$"),
    labels.index("Empirical advantage"),
    labels.index(r"Theory advantage ($\mu$-GDP)"),
    labels.index("FPR (α)"),
    labels.index("FNR (β)"),
]
ax.legend([handles[i] for i in order], [labels[i] for i in order],
          loc="upper right", fontsize=11)

plt.grid(alpha=0.12)
plt.tight_layout()
plt.show()
# fig.savefig("../data/valid.pdf")


In [ ]:
results_all = {}

# baseline
baseline_ww_out, baseline_ww_in = features["out"][:, 9], features["in"][:, 9]
results, _ = mu_lower_from_two_groups(baseline_ww_out, baseline_ww_in)
results_all["Baseline"] = results

# threshold_selection -- max_mu_hat
results = run_audit(features["out"],
                    features["in"],
                    n_train=N_TRAIN,
                    n_valid=N_VALID,
                    n_test=N_TEST,
                    threshold_selection="max_mu_hat",
                    random_state=13)
results_all[r"$\hat{\mu}$"] = results['test']['point']['mu_lower']

# ci_method -- bonferroni_cp
results = run_audit(features["out"],
                    features["in"],
                    n_train=N_TRAIN,
                    n_valid=N_VALID,
                    n_test=N_TEST,
                    ci_method="bonferroni_cp",
                    random_state=13)
results_all["Clopper–Pearson"] = results['test']['point']['mu_lower']

# D_out size -- 2
# with open('../data/features_2.pkl', 'rb') as handle:
#     features_2 = pickle.load(handle)
# results = run_audit(features_2["out"],
#                     features_2["in"],
#                     n_train=N_TRAIN,
#                     n_valid=N_VALID,
#                     n_test=N_TEST,
#                     random_state=13)
# results_all["$|D_{out}| = 2$"] = results['test']['point']['mu_lower']
results_all["$|D_{out}| = 2$"] = 0.29832043950664605

# classifier -- random_forest
results = run_audit(features["out"],
                    features["in"],
                    n_train=N_TRAIN,
                    n_valid=N_VALID,
                    n_test=N_TEST,
                    classifier="random_forest",
                    random_state=13)
results_all["Random Forest"] = results['test']['point']['mu_lower']

# threat model -- black-box
results = run_audit(features["out"][:, :8],
                    features["in"][:, :8],
                    n_train=N_TRAIN,
                    n_valid=N_VALID,
                    n_test=N_TEST,
                    random_state=13)
results_all["Black-box"] = results['test']['point']['mu_lower']

# threat model -- white-box
results = run_audit(features["out"][:, 8:],
                    features["in"][:, 8:],
                    n_train=N_TRAIN,
                    n_valid=N_VALID,
                    n_test=N_TEST,
                    random_state=13)
results_all["White-box"] = results['test']['point']['mu_lower']

# default
results_all["Default"] = default_results['test']['point']['mu_lower']

In [ ]:
# results = {
#     "Baseline": 0.2601335097836669,
#     r"$\hat{\mu}$": 0.0,
#     "Clopper–Pearson": 0.2796166719825678,
#     r"$|D_{out}| = 2$": 0.29832043950664605,
#     "Random Forest": 0.32397972624803917,
#     "Black-box": 0.3901927720079095,
#     "White-box": 0.4150704207293557,
#     "Default": 0.42617713009324837,
# }

In [ ]:
names = list(results_all.keys())
values = list(results_all.values())

fig = plt.figure(figsize=(7, 5))

bars = plt.bar(names, values, color="black", label="Empirical audit")

# ---- highlight baseline ----
for bar, name in zip(bars, names):
    if name == "Default":
        bar.set_edgecolor("red")
        bar.set_linewidth(5)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 0.003, f"{h:.2f}",
             ha="center", va="bottom", fontsize=11)

# ---- theoretical line ----
plt.axhline(
    IMPLIED_MU,
    linestyle="-",
    linewidth=3.0,
    color="red",
    alpha=0.95,
    label=r"Theory $\mu$ (via $\rho$-zCDP)",
)

plt.legend(
    loc="upper left",
    bbox_to_anchor=(0.01, 0.95),  # move down a bit
    fontsize=12
)

plt.ylabel(r"$\mu_{emp}$", fontsize=12)
# plt.xlabel("Ablation setting")

plt.grid(axis="y", linestyle="--", alpha=0.12)
plt.xticks(rotation=30, fontsize=12)

plt.tight_layout()
plt.show()
# fig.savefig("../data/abl.pdf")